In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Gold Layer: Daily Financial Reconciliation (`gold.daily_financial_reconciliation`)
# MAGIC 
# MAGIC - **Sources**: `silver.fct_financial_ledger_cdc`
# MAGIC - **Target**: `gold.daily_financial_reconciliation` (Managed Delta table in Unity Catalog)
# MAGIC - **Key Analytics**:
# MAGIC   - Reconciles total authorized GMV, captured cash, applied discounts, taxes, and refunds.
# MAGIC   - Computes Net Cash Settled per business day.



In [0]:
# COMMAND ----------

from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# COMMAND ----------

# 1. Read Silver Financial CDC Fact
df_ledger = spark.table("silver.fct_financial_ledger_cdc")

# COMMAND ----------



In [0]:
# 2. Aggregate Daily Financial Metrics
df_financial_recon = (
    df_ledger
    .filter(F.col("order_date").isNotNull())
    .groupBy("order_date", "payment_method")
    .agg(
        F.countDistinct("order_id").alias("total_transactions"),
        F.round(F.sum("gross_amount"), 2).alias("total_gmv"),
        F.round(F.sum("discount_applied"), 2).alias("total_discounts_granted"),
        F.round(F.sum("tax_amount"), 2).alias("total_tax_collected"),
        F.round(
            F.sum(F.when(F.col("payment_status") == "REFUNDED", F.col("net_settled_amount")).otherwise(0.0)),
            2
        ).alias("total_refunded_amount"),
        F.round(
            F.sum(F.when(F.col("payment_status") == "SETTLED", F.col("net_settled_amount")).otherwise(0.0)),
            2
        ).alias("net_cash_settled")
    )
    .withColumn(
        "net_operating_revenue",
        F.round(F.col("net_cash_settled") - F.col("total_refunded_amount"), 2)
    )
    .withColumn("_transformed_at", F.current_timestamp())
)

# COMMAND ----------

# 3. Write to Unity Catalog managed table and optimize
(
    df_financial_recon.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.daily_financial_reconciliation")
)

spark.sql("OPTIMIZE gold.daily_financial_reconciliation ZORDER BY (order_date, payment_method)")

print("SUCCESS: gold.daily_financial_reconciliation registered successfully.")

# COMMAND ----------

# 4. Preview Output
display(
    spark.table("gold.daily_financial_reconciliation")
    .orderBy(F.col("order_date").desc())
    .limit(10)
)